In [1]:
import pandas as pd
import glob
import pyarrow as pa
import pyarrow.parquet as pq

BASE_DIR = "/workspaces/Project/CSDS555-ResAI-Final-Project-Research/data/input/dataset"

paths = glob.glob(f"{BASE_DIR}/**/*.parquet", recursive=True)

print(f"Found {len(paths)} parquet files")

dfs = []
common_fields = set()

# First pass — discover ALL fields across files
for p in paths:
    schema = pq.read_schema(p)
    common_fields |= set(schema.names)

common_fields = sorted(common_fields)
print("Unified schema:", common_fields)

# Second pass — load & align schemas
for p in paths:
    df = pd.read_parquet(p)

    # Add missing columns as null to match unified schema
    for col in common_fields:
        if col not in df.columns:
            df[col] = None

    df = df[common_fields]   # reorder columns
    dfs.append(df)

# Final merged DF
df_all = pd.concat(dfs, ignore_index=True)
df.groupby("UUID")

print(df_all.head())
print(df_all.shape)

Found 31 parquet files
Unified schema: ['UUID', 'identity_A', 'identity_B', 'scenario_id']
       UUID  identity_A  identity_B  scenario_id
0  11612000         621        1067            0
1  11612001         621        1067            1
2  11612002         621        1067            2
3  11612003         621        1067            3
4  11612004         621        1068            0
(87347716, 4)


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2903000 entries, 0 to 2902999
Data columns (total 4 columns):
 #   Column       Dtype
---  ------       -----
 0   UUID         int64
 1   identity_A   int64
 2   identity_B   int64
 3   scenario_id  int64
dtypes: int64(4)
memory usage: 88.6 MB


: 